# Data Preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import glob

# notebook paths (verify on your side)
BASE_DIR = "/content/drive/MyDrive/AMEX Challenge"
TRAIN_CSV = f"{BASE_DIR}/train_data.csv"
TEST_CSV  = f"{BASE_DIR}/test_data.csv"
TRAIN_PARQUET_DIR = f"{BASE_DIR}/train_parquet"
TEST_PARQUET_DIR  = f"{BASE_DIR}/test_parquet"

AGG_DATA_PATH = f"{BASE_DIR}/train_aggregated_v1.parquet"
LABELS_PATH   = f"{BASE_DIR}/train_labels.csv"

CHUNKSIZE = 500_000 # This amount seems okay for COllab

> We have used highly optimized Pre-processing Pipeline specifically designed to handle the Amex dataset scale within the memory limits of Google Colab.

In [ ]:
# import numpy as np
# import pandas as pd. # additional runtime import for new session
# import os

# Define the mappings for string-based categories to ensure
# they are consistent across all chunks.


d_63_map = {'CR': 0, 'CO': 1, 'CL': 2, 'XZ': 3, 'XM': 4, 'XL': 5}
d_64_map = {'-1': 0, 'O': 1, 'R': 2, 'U': 3}

#Note: Manual dictionaries that map the only two string-based columns in the dataset to integers.
#Note: Using a manual map instead of a dynamic LabelEncoder ensures that 'CR' is always 0, even if we process the data in separate chunks.
#This prevents "ID mismatch" between our training and testing sets.


def process_features_chunk(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies flooring-based denoising and handles string/numeric categoricals.
    """
    # 1. Date Conversion
    if "S_2" in df.columns:
        df["S_2"] = pd.to_datetime(df["S_2"])

    # 2. Denoising: Floor to 2 decimal places

    # Note: It multiplies the values by 100, chops off the decimals (flooring), and divides by 100.

    # Since Amex added noise in the range of [0, 0.01], this operation effectively strips away that jitter,
      # returning the features to their original discrete "steps."

    float_cols = df.select_dtypes(include=['float32', 'float64']).columns
    df[float_cols] = np.floor(df[float_cols] * 100) / 100

    # 3. Categorical Handling (The Fix)
    # Mapping strings to integers before downcasting

    # Note: Uses the map created earlier to replace strings with integers.
    # fillna(-1) handles missing data by assigning it a unique code, and astype('int8') compresses the
    # data to the smallest possible integer size (1 byte per row).

    if 'D_63' in df.columns:
        df['D_63'] = df['D_63'].map(d_63_map).fillna(-1).astype('int8')
    if 'D_64' in df.columns:
        df['D_64'] = df['D_64'].map(d_64_map).fillna(-1).astype('int8')

    # Note: For the other 9 categorical columns (which are already numeric in the CSV), it ensures they are properly formatted as integers.
    cat_cols = ["B_30", "B_38", "D_114", "D_116", "D_117", "D_120", "D_126", "D_66", "D_68"]
    for col in cat_cols:
        if col in df.columns:
            # Converting to numeric first to handle cases where they might be strings
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1).astype('int8')

    # 4. Memory Optimization
    # Downcast floats to float32 to save 50% RAM in Colab
    df[float_cols] = df[float_cols].astype('float32')

    return df

In [ ]:
def csv_to_parquet_in_chunks(csv_path, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    reader = pd.read_csv(csv_path, chunksize=CHUNKSIZE)

    #Note: chunksize=CHUNKSIZE: Instead of returning a standard DataFrame,
    # pd.read_csv returns a "TextFileReader" object.
    # It will only pull a specific number of rows into RAM at a time.

    for i, chunk in enumerate(reader):

        try:

            processed_chunk = process_features_chunk(chunk)
            chunk_name = f"part_{i:05d}.parquet"
            processed_chunk.to_parquet(os.path.join(output_dir, chunk_name), index=False)
            print(f"Chunk {i} processed and saved.")

            # del: This is to explicitly tell Python to forget the data in that chunk.

            #This is critical in Colab to ensure the RAM is "emptied" before the next chunk is loaded.

            del processed_chunk

        except Exception as e:

            print(f"Error in chunk {i}: {e}")

            return chunk

csv_to_parquet_in_chunks(TRAIN_CSV, TRAIN_PARQUET_DIR)
csv_to_parquet_in_chunks(TEST_CSV, TEST_PARQUET_DIR)

✅ Chunk 0 processed and saved.
✅ Chunk 1 processed and saved.
✅ Chunk 2 processed and saved.
✅ Chunk 3 processed and saved.
✅ Chunk 4 processed and saved.
✅ Chunk 5 processed and saved.
✅ Chunk 6 processed and saved.
✅ Chunk 7 processed and saved.
✅ Chunk 8 processed and saved.
✅ Chunk 9 processed and saved.
✅ Chunk 10 processed and saved.
✅ Chunk 11 processed and saved.
✅ Chunk 0 processed and saved.
✅ Chunk 1 processed and saved.
✅ Chunk 2 processed and saved.
✅ Chunk 3 processed and saved.
✅ Chunk 4 processed and saved.
✅ Chunk 5 processed and saved.
✅ Chunk 6 processed and saved.
✅ Chunk 7 processed and saved.
✅ Chunk 8 processed and saved.
✅ Chunk 9 processed and saved.
✅ Chunk 10 processed and saved.
✅ Chunk 11 processed and saved.
✅ Chunk 12 processed and saved.
✅ Chunk 13 processed and saved.
✅ Chunk 14 processed and saved.
✅ Chunk 15 processed and saved.
✅ Chunk 16 processed and saved.
✅ Chunk 17 processed and saved.
✅ Chunk 18 processed and saved.
✅ Chunk 19 processed and sav

In [ ]:
# DO NOT RUN! <<seule fiche aggrege>>

import os

# Configuration
BASE_DIR = "/content/drive/MyDrive/AMEX Challenge"
INPUT_DIR = f"{BASE_DIR}/train_parquet"
OUTPUT_FILE = f"{BASE_DIR}/train_aggregated_v1.parquet"
LABELS_FILE = f"{BASE_DIR}/train_labels.csv"

def get_agg_features(df):
    """
    Calculates mean, std, min, max, and last for numerical columns,
    plus 'velocity' (diff) features.
    """
    # Identify column groups based on the Amex Plan
    all_cols = df.columns.tolist()
    cat_features = ["B_30", "B_38", "D_114", "D_116", "D_117", "D_120", "D_126", "D_63", "D_64", "D_66", "D_68"]
    num_features = [c for c in all_cols if c not in cat_features + ['customer_ID', 'S_2']]

    # 1. Numerical Aggregations (The "History" Summary)
      # The "Last" column: Captures the current financial state.
    agg_num = df.groupby("customer_ID")[num_features].agg(['mean', 'std', 'min', 'max', 'last'])
    agg_num.columns = ['_'.join(x) for x in agg_num.columns]

    # 2. Velocity Features (Last - First) ... also the trend indicator
    # We calculate the change over the 13-month period
    first_values = df.groupby("customer_ID")[num_features].first()
    last_values = df.groupby("customer_ID")[num_features].last()
    diff_features = last_values - first_values
    diff_features.columns = [f"{c}_diff" for c in diff_features.columns]

    # 3. Categorical Aggregations (Mode and Last)
    agg_cat = df.groupby("customer_ID")[cat_features].agg(['last', 'nunique'])
    agg_cat.columns = ['_'.join(x) for x in agg_cat.columns]

    return pd.concat([agg_num, diff_features, agg_cat], axis=1)

# Process all chunks
all_files = sorted(glob.glob(f"{INPUT_DIR}/*.parquet"))
agg_list = []

for i, f in enumerate(all_files):
    print(f"Aggregating chunk {i}...")
    chunk = pd.read_parquet(f)
    agg_list.append(get_agg_features(chunk))
    del chunk

# Combine and save
final_df = pd.concat(agg_list).groupby(level=0).last() # Merge duplicates if a customer spans chunks
final_df.to_parquet(OUTPUT_FILE)
print(f"Aggregated features saved to {OUTPUT_FILE}")

Aggregating chunk 0...
Aggregating chunk 1...
Aggregating chunk 2...
Aggregating chunk 3...
Aggregating chunk 4...
Aggregating chunk 5...
Aggregating chunk 6...
Aggregating chunk 7...
Aggregating chunk 8...
Aggregating chunk 9...
Aggregating chunk 10...
Aggregating chunk 11...
✅ Aggregated features saved to /content/drive/MyDrive/AMEX Challenge/train_aggregated_v1.parquet


Now that the data is ready, we move to training. Per plan, we start with the Tree Experts (GBDTs) using the aggregated data.


1. The Amex Metric (Custom Evaluation)
We must use the specific competition metric (Gini + Top 4% Capture) as your early-stopping criteria. (based on knowledge gathered from leaderboard solutions)

This implementation uses 5-Fold Stratified Cross-Validation and the Target Weight of 20 for the negative class.

In [ ]:
import os

# Configuration
BASE_DIR = "/content/drive/MyDrive/AMEX Challenge"
INPUT_DIR = f"{BASE_DIR}/train_parquet"
TEMP_AGG_DIR = f"{BASE_DIR}/temp_aggregates"

# FINAL_OUTPUT_FILE is the "master table" (1 row per customer) used for training Tree models.
FINAL_OUTPUT_FILE = f"{BASE_DIR}/train_aggregated_v1.parquet"

os.makedirs(TEMP_AGG_DIR, exist_ok=True)

def get_agg_features(df):
    """
    Calculates features based on the Amex Plan:
    Mean, Std, Min, Max, and Last for numericals.
    Difference (Velocity) features.
    """
    # Explicitly separate categories to avoid performing mathematical operations (like 'mean') on IDs.
    cat_features = ["B_30", "B_38", "D_114", "D_116", "D_117", "D_120", "D_126", "D_63", "D_64", "D_66", "D_68"]
    num_features = [c for c in df.columns if c not in cat_features + ['customer_ID', 'S_2']]

    # Aggregations: Collapse 13 rows into 1 by calculating the "Story" of the customer.
    # Mean/Last are baseline; Std captures volatility (how much the customer's balance fluctuates).
    agg_df = df.groupby("customer_ID")[num_features].agg(['mean', 'std', 'min', 'max', 'last'])
    # Rename columns to 'Feature_mean', 'Feature_last', etc., to keep the table flat and organized.
    agg_df.columns = ['_'.join(x) for x in agg_df.columns]

    # Velocity: Measures the 'trajectory' of the financial behavior.
    # A positive diff in debt (B_*) columns suggests a customer is spiraling deeper into debt.
    first_vals = df.groupby("customer_ID")[num_features].first()
    last_vals = df.groupby("customer_ID")[num_features].last()
    for col in num_features:
        agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]

    # Categoricals: 'last' tells us the current status; 'nunique' tells us if they frequently change status.
    agg_cat = df.groupby("customer_ID")[cat_features].agg(['last', 'nunique'])
    agg_cat.columns = ['_'.join(x) for x in agg_cat.columns]

    # Combine numerical summaries, velocity trends, and categorical statuses into one wide row.
    return pd.concat([agg_df, agg_cat], axis=1)

# STEP 1: Process and save temporary partial aggregates
# We process one shard at a time to stay under the 12GB RAM limit of Colab Free.
all_files = sorted(glob.glob(f"{INPUT_DIR}/*.parquet"))
for i, f in enumerate(all_files):
    print(f"Processing chunk {i}...")
    chunk = pd.read_parquet(f)
    agg_chunk = get_agg_features(chunk)
    # Saving to disk immediately releases the RAM used by this chunk's calculations.
    agg_chunk.to_parquet(f"{TEMP_AGG_DIR}/part_{i}.parquet")
    # Explicitly delete objects to ensure the garbage collector clears memory.
    del chunk, agg_chunk

# STEP 2: Merge the partial aggregates serially
# This step handles 'Fragmented Customers' who might have months split across different input shards.
print("Merging partial aggregates...")
temp_files = glob.glob(f"{TEMP_AGG_DIR}/*.parquet")
final_df = None

for f in temp_files:
    part = pd.read_parquet(f)
    if final_df is None:
        final_df = part
    else:
        # Concatenate and GroupBy ensures we only have ONE row per unique customer ID in the end.
        # .last() picks the most up-to-date aggregation if duplicates exist.
        final_df = pd.concat([final_df, part]).groupby(level=0).last()
    del part

# Save the final engineered dataset. This file is roughly 1,000 columns wide and 458k rows deep.
final_df.to_parquet(FINAL_OUTPUT_FILE)
print(f"✅ Final aggregated file saved: {FINAL_OUTPUT_FILE}")

Processing chunk 0...


/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. 

Processing chunk 1...


/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. 

Processing chunk 2...


...

Processing chunk 10...


/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. 

Processing chunk 11...


/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]
/tmp/ipython-input-2615296817.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. 

Merging partial aggregates...
✅ Final aggregated file saved: /content/drive/MyDrive/AMEX Challenge/train_aggregated_v1.parquet


In [ ]:
# def get_agg_features(df):
#     """
#     Calculates features based on the Amex Plan:
#     Mean, Std, Min, Max, and Last for numericals.
#     Difference (Velocity) features.
#     """
#     cat_features = ["B_30", "B_38", "D_114", "D_116", "D_117", "D_120", "D_126", "D_63", "D_64", "D_66", "D_68"]
#     num_features = [c for c in df.columns if c not in cat_features + ['customer_ID', 'S_2']]

#     # Aggregations: Mean, Std, Min, Max, Last
#     agg_df = df.groupby("customer_ID")[num_features].agg(['mean', 'std', 'min', 'max', 'last'])
#     agg_df.columns = ['_'.join(x) for x in agg_df.columns]

#     # Velocity: Last - First
#     first_vals = df.groupby("customer_ID")[num_features].first()
#     last_vals = df.groupby("customer_ID")[num_features].last()
#     for col in num_features:
#         agg_df[f"{col}_diff"] = last_vals[col] - first_vals[col]

#     # Categoricals: Last and Nunique
#     agg_cat = df.groupby("customer_ID")[cat_features].agg(['last', 'nunique'])
#     agg_cat.columns = ['_'.join(x) for x in agg_cat.columns]

#     return pd.concat([agg_df, agg_cat], axis=1)